# AD688 Group 3 - Step 2 Filtering: Insurance Carriers, All Roles (PySpark on AWS EC2)

**Track:** Team project, Step 2 (Data Preparation and Market Baseline).

**Engine:** PySpark, running on the AWS EC2 instance (the class-proper toolchain).

**Goal:** filter the job postings down to the Insurance Carriers industry (NAICS 5241),keeping every role, not just data roles, and flag which postings match our data-role scope. 

This is the team's master dataset going forward. Everything is included, and
`IS_DATA_ROLE_MATCH` column marks the ones relevant to the data-role analysis, so the team can sort/filter to that subset in Excel or widen the view as needed.

Run the cells top to bottom. Each one prints its output right below it. 


## 1. Start Spark and load the data

Spark is the query engine. We point it at the 17 Parquet files and load them as one table. 

In [47]:
# Start a local Spark session - this is the engine that reads and queries the data.
# local[1] = use 1 CPU core, 512m = cap memory at 512MB. Both are set low on purpose
# because this notebook runs on a small EC2 instance with under 1GB of RAM.
import glob, os
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("AD688-Step2-Insurance")
         .master("local[1]")
         .config("spark.driver.memory", "512m")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")  # quiet down Spark's own log noise

# Find every Parquet file in the data folder and load them all as one table.
files = glob.glob("data/MET_CareerCompass_2026/*.parquet")
df = spark.read.parquet(*files)

print("Parquet files loaded:", len(files))
print("Total postings:", df.count())
print("Columns:", len(df.columns))

Parquet files loaded: 17


Total postings: 165386


Columns: 115


## 2. Register the data as a SQL table

This lets us query the postings with plain SQL and name the table `jobs`.

In [48]:
# Register the DataFrame as a temporary SQL table named "jobs" so the next cell can
# query it with plain SQL (SELECT / WHERE) instead of PySpark method chains.
df.createOrReplaceTempView("jobs")
print("Table 'jobs' is ready to query.")

Table 'jobs' is ready to query.


## 3. Filter to Insurance Carriers (NAICS 5241), flagged for data-role titles

Keep every posting in the Insurance Carriers industry (NAICS code 5241),
regardless of job title - nothing gets dropped, this is still the full industry cross-section. 

The one addition is a new column, `IS_DATA_ROLE_MATCH`, placed right
after `TITLE_CLEAN`: it's 1 for postings whose title matches our data-role keyword set (data engineer, data scientist, data analyst / BI, data architect / DBA, data governance, ML engineer, etc), and 0 for everything else. 

Sorting or filtering on this column in Excel is how the team can hone in on data roles specifically, without losing the rest of the dataset to work from.

In [49]:
# This is still the full Insurance Carrier (NAICS 5241) slice - every role, nothing
# filtered out. The only addition is IS_DATA_ROLE_MATCH, a 1/0 flag placed right after
# TITLE_CLEAN, using our data-role keyword regex. Rows flagged 1 are the data-role subset.
query = """
SELECT
  -- Core identifiers and posting metadata
  ID, POSTED, EXPIRED, DURATION, URL,
  -- Job title fields (raw text plus Lightcast's cleaned versions)
  TITLE_RAW, TITLE_NAME, TITLE_CLEAN,
  -- Flag: 1 if this title matches the data-role keyword set, 0 otherwise. Does not
  -- drop any rows - just marks which ones are in the data-role scope.
  CASE WHEN lower(TITLE_RAW) RLIKE 'data engineer|analytics engineer|etl|data pipeline|data platform|big data|ml engineer|machine learning engineer|data architect|database|data scientist|data science|data analyst|business intelligence| bi |data governance|data management'
       THEN 1 ELSE 0 END AS IS_DATA_ROLE_MATCH,
  -- Employer info
  COMPANY_NAME, COMPANY_IS_STAFFING, EMPLOYMENT_TYPE_NAME,
  -- Experience and education requirements
  MIN_YEARS_EXPERIENCE, MAX_YEARS_EXPERIENCE, IS_INTERNSHIP,
  MIN_EDULEVELS_NAME, MAX_EDULEVELS_NAME,
  -- Pay
  SALARY, SALARY_FROM, SALARY_TO, ORIGINAL_PAY_PERIOD,
  -- Location and remote status
  REMOTE_TYPE_NAME, LOCATION, CITY_NAME, STATE_NAME, MSA_NAME,
  -- Industry and occupation codes
  NAICS_2022_4, SOC_2021_4_NAME, ONET_NAME,
  -- Skills and certifications requested in the posting
  SKILLS_NAME, SPECIALIZED_SKILLS_NAME, COMMON_SKILLS_NAME,
  SOFTWARE_SKILLS_NAME, CERTIFICATIONS_NAME
FROM jobs
WHERE NAICS_2022_4 = '5241'
ORDER BY TITLE_NAME, TITLE_RAW
"""
insurance = spark.sql(query)
insurance.createOrReplaceTempView("insurance_jobs")
print("Insurance Carrier (NAICS 5241) postings:", insurance.count())
print("Rows flagged IS_DATA_ROLE_MATCH = 1:", insurance.filter("IS_DATA_ROLE_MATCH = 1").count())


[Stage 4:=======================================>                   (2 + 1) / 3]



Insurance Carrier (NAICS 5241) postings: 1895


Rows flagged IS_DATA_ROLE_MATCH = 1: 27


## 4. See the results

A preview of the postings, and the most common job titles in this slice so the team
can get a feel for what's in here before deciding how to narrow it down.

In [50]:
# Preview the postings, including the new flag, and the most common job titles.
insurance.select("TITLE_RAW", "TITLE_NAME", "IS_DATA_ROLE_MATCH", "COMPANY_NAME",
                  "STATE_NAME", "REMOTE_TYPE_NAME", "SALARY_FROM").show(30, truncate=False)

print("Most common job titles (TITLE_NAME), top 25:")
insurance.groupBy("TITLE_NAME").count().orderBy("count", ascending=False).show(25, truncate=False)

+------------------------------------------------------------------------+------------------------+------------------+---------------------------+---------------------+----------------+-----------+
|TITLE_RAW                                                               |TITLE_NAME              |IS_DATA_ROLE_MATCH|COMPANY_NAME               |STATE_NAME           |REMOTE_TYPE_NAME|SALARY_FROM|
+------------------------------------------------------------------------+------------------------+------------------+---------------------------+---------------------+----------------+-----------+
|Accountant                                                              |Accountants and Auditors|0                 |Jobs.danaher.com           |California           |Unknown         |           |
|Accountant                                                              |Accountants and Auditors|0                 |Bannerlife                 |Maryland             |Unknown         |           |
|Accountan

+------------------------------------------------------------------------------------------------+-----+
|TITLE_NAME                                                                                      |count|
+------------------------------------------------------------------------------------------------+-----+
|Management Analysts                                                                             |643  |
|Project Management Specialists                                                                  |472  |
|Actors                                                                                          |244  |
|Marketing Managers                                                                              |87   |
|Information Security Analysts                                                                   |66   |
|Data Scientists                                                                                 |44   |
|Software Developers                                   

## 5. Export for the team (Excel + CSV)

This is bigger than the 27-row data-role cut (about 1,900 rows), but still small enough
to pull into pandas and write out directly.

In [51]:
# Pull the full Insurance Carrier slice into pandas and write it out as both Excel
# and CSV, so the team has one shared file to review and narrow down from.
os.makedirs("outputs/step2", exist_ok=True)
pdf = insurance.toPandas()

xlsx_path = "outputs/step2/insurance_5241_all_postings.xlsx"
csv_path  = "outputs/step2/insurance_5241_all_postings.csv"
pdf.to_excel(xlsx_path, index=False)
pdf.to_csv(csv_path, index=False)

print("Rows exported:", len(pdf))
print("Excel:", os.path.abspath(xlsx_path))
print("CSV:  ", os.path.abspath(csv_path))

/home/ubuntu/ad688/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/ubuntu/ad688/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)



[Stage 14:======================================>                   (2 + 1) / 3]



Rows exported: 1895
Excel: /home/ubuntu/ad688/outputs/step2/insurance_5241_all_postings.xlsx
CSV:   /home/ubuntu/ad688/outputs/step2/insurance_5241_all_postings.csv


## 6. What this produced

- The team's master dataset: the full Insurance Carrier (NAICS 5241) posting set, every
  role, nothing filtered out.
- A new `IS_DATA_ROLE_MATCH` column (1/0) right after `TITLE_CLEAN`, flagging postings
  that match our data-role keyword set. Sort or filter on that column in Excel/CSV to
  jump straight to the data roles, or clear the filter to see everything else in the
  industry.
- Two export files in `outputs/step2/` (Excel and CSV): `insurance_5241_all_postings`.
  This is the file the team should use going forward.

When done, stop Spark to free memory:

In [52]:
# Release Spark's memory now that we're done with it.
spark.stop()
print("Spark stopped.")

Spark stopped.
